In [0]:
# Create widgets for landing_ts and table_name
dbutils.widgets.text("landing_timeStamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_key", "", "Merge Key")
dbutils.widgets.text("load_type", "", "Load Type")

# Retrieve widget values into variables
landing_timeStamp = dbutils.widgets.get("landing_timeStamp")
table_name = dbutils.widgets.get("table_name")
merge_key = dbutils.widgets.get("merge_key")
load_type = dbutils.widgets.get("load_type")

#why we need current timestamp is because we need to check the latest to fill in the bronze

In [0]:
bronze_table = f"adb_caresync_prithwish.bronze.{table_name}"
filter_condition = f"landing_timeStamp = '{landing_timeStamp}'"

print("=" * 80)
print("STEP 1: Reading bronze data")
print("=" * 80)
print(f"Parameters -> table_name: '{table_name}', landing_timeStamp: '{landing_timeStamp}', load_type: '{load_type}', merge_key: '{merge_key}'")
print(f"Source table     : {bronze_table}")
print(f"Filter condition : {filter_condition}")
print(f"Equivalent query : SELECT * FROM {bronze_table} WHERE {filter_condition}")

bronze_df=spark.read.format("delta").table(bronze_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"Records read from bronze: {bronze_count}")
if bronze_count == 0:
    print("WARNING: No records found for this landing timestamp. Check the landing_timeStamp value.")
#filtering for landing timestamp
# .load only conatains storage path while .table contains table name
display(bronze_df)

In [0]:
#adding two more cloumns to bronze table
from pyspark.sql import functions as F
print("=" * 80)
print("STEP 2: Adding audit columns")
print("=" * 80)
bronze_df_2=bronze_df.withColumn("insert_timeStame",F.current_timestamp()).withColumn("updated_timeStamp",F.current_timestamp())
print("Added columns    : insert_timeStame = current_timestamp(), updated_timeStamp = current_timestamp()")
print(f"Final column list ({len(bronze_df_2.columns)}): {bronze_df_2.columns}")

#purpose of update timestamp is building logic while merge operations
#display(bronze_df_2)

In [0]:
from delta.tables import DeltaTable

silver_table = f"adb_caresync_prithwish.silver.{table_name}"
load_type_norm = load_type.strip().upper()
column_list = ", ".join(bronze_df_2.columns)

print("=" * 80)
print(f"STEP 3: Loading into silver | load_type = '{load_type_norm}'")
print("=" * 80)
print(f"Target table : {silver_table}")
target_exists = spark.catalog.tableExists(silver_table)
print(f"Target exists: {target_exists}")
if target_exists:
    print(f"Target record count BEFORE load: {spark.table(silver_table).count()}")

if load_type_norm == "FULL":
    # Full load: overwrite the silver table
    print("Mode: FULL -> overwriting target table with the current batch")
    print(f"Equivalent query:\nCREATE OR REPLACE TABLE {silver_table} AS\nSELECT {column_list}\nFROM <bronze batch>")
    bronze_df_2.write.mode("overwrite").format("delta").saveAsTable(silver_table)
    print("FULL load completed.")

elif load_type_norm == "APPEND":
    # Append: add the new batch to the silver table (creates it if missing)
    print("Mode: APPEND -> appending current batch to target table")
    print(f"Equivalent query:\nINSERT INTO {silver_table} ({column_list})\nSELECT {column_list}\nFROM <bronze batch>")
    bronze_df_2.write.mode("append").format("delta").saveAsTable(silver_table)
    print("APPEND load completed.")

elif load_type_norm == "MERGE":
    # Merge keys can be a single column or comma-separated list, e.g. "hospital_id" or "id,region"
    print("Mode: MERGE -> update existing records and insert new records")
    merge_keys = [k.strip() for k in merge_key.split(",") if k.strip()]
    print(f"Merge keys parsed: {merge_keys}")
    if not merge_keys:
        raise ValueError("merge_key widget must be provided when load_type is MERGE")
    missing = [k for k in merge_keys if k not in bronze_df_2.columns]
    if missing:
        raise ValueError(f"Merge key column(s) not found in source: {missing}")
    print("Merge key validation passed.")

    # Keep one row per key so the MERGE does not fail on multiple source matches
    source_count = bronze_df_2.count()
    source_df = bronze_df_2.dropDuplicates(merge_keys)
    dedup_count = source_df.count()
    print(f"Source records: {source_count} | After de-duplication on {merge_keys}: {dedup_count} | Duplicates removed: {source_count - dedup_count}")

    if not target_exists:
        # First run: nothing to merge into, so create the table
        print("Target table does not exist -> creating it from the source batch (no merge needed)")
        print(f"Equivalent query:\nCREATE TABLE {silver_table} AS\nSELECT {column_list}\nFROM <bronze batch, de-duplicated on {', '.join(merge_keys)}>")
        source_df.write.format("delta").saveAsTable(silver_table)
        print("Target table created.")
    else:
        target_table = DeltaTable.forName(spark, silver_table)
        merge_condition = " AND ".join([f"target.`{key}` = source.`{key}`" for key in merge_keys])

        # On update keep the original insert timestamp; refresh everything else
        update_set = {f"`{column}`": f"source.`{column}`" for column in source_df.columns if column != "insert_timeStame"}
        insert_set = {f"`{column}`": f"source.`{column}`" for column in source_df.columns}

        print(f"Merge condition : {merge_condition}")
        print(f"Columns updated on match ({len(update_set)}): {list(update_set.keys())}")
        print(f"Columns inserted on no match ({len(insert_set)}): {list(insert_set.keys())}")

        # Build the equivalent SQL MERGE statement for logging
        update_clause = ",\n    ".join([f"target.{col} = {val}" for col, val in update_set.items()])
        insert_cols = ", ".join(insert_set.keys())
        insert_vals = ", ".join(insert_set.values())
        merge_sql = (
            f"MERGE INTO {silver_table} AS target\n"
            f"USING <bronze batch, de-duplicated on {', '.join(merge_keys)}> AS source\n"
            f"ON {merge_condition}\n"
            f"WHEN MATCHED THEN UPDATE SET\n    {update_clause}\n"
            f"WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
            f"VALUES ({insert_vals})"
        )
        print("Equivalent query:")
        print(merge_sql)

        print("Executing MERGE...")
        (target_table.alias("target")
            .merge(source_df.alias("source"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute())

        # Report what the MERGE did, from the Delta table history
        merge_metrics = target_table.history(1).select("operationMetrics").collect()[0][0]
        print("MERGE completed. Metrics:")
        print(f"  Rows inserted : {merge_metrics.get('numTargetRowsInserted')}")
        print(f"  Rows updated  : {merge_metrics.get('numTargetRowsUpdated')}")
        print(f"  Rows deleted  : {merge_metrics.get('numTargetRowsDeleted')}")
        print(f"  Source rows   : {merge_metrics.get('numSourceRows')}")

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

print(f"Target record count AFTER load: {spark.table(silver_table).count()}")
print("=" * 80)

In [0]:
%sql
-- INACTIVE: SQL equivalent of the Python MERGE branch, kept for reference only.
-- Example written for table_name = 'hospitals' with merge key hospital_id.
-- To use it: uncomment, adjust the table/merge key/columns, and run.

-- MERGE INTO adb_caresync_prithwish.silver.hospitals AS target
-- USING (
--   SELECT
--     hospital_id, hospital_name, city, state, hospital_type, bed_count,
--     created_at, updated_at, landing_timeStamp, insert_timeStamp,
--     current_timestamp() AS insert_timeStame,
--     current_timestamp() AS updated_timeStamp
--   FROM adb_caresync_prithwish.bronze.hospitals
--   WHERE landing_timeStamp = :landing_timeStamp
--   -- keep one row per merge key (same role as dropDuplicates in Python)
--   QUALIFY ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY updated_at DESC) = 1
-- ) AS source
-- ON target.hospital_id = source.hospital_id
-- WHEN MATCHED THEN UPDATE SET
--   -- every column except insert_timeStame, so the original insert time is preserved
--   target.hospital_name     = source.hospital_name,
--   target.city              = source.city,
--   target.state             = source.state,
--   target.hospital_type     = source.hospital_type,
--   target.bed_count         = source.bed_count,
--   target.created_at        = source.created_at,
--   target.updated_at        = source.updated_at,
--   target.landing_timeStamp = source.landing_timeStamp,
--   target.insert_timeStamp  = source.insert_timeStamp,
--   target.updated_timeStamp = source.updated_timeStamp
-- WHEN NOT MATCHED THEN INSERT (
--   hospital_id, hospital_name, city, state, hospital_type, bed_count,
--   created_at, updated_at, landing_timeStamp, insert_timeStamp,
--   insert_timeStame, updated_timeStamp
-- ) VALUES (
--   source.hospital_id, source.hospital_name, source.city, source.state, source.hospital_type, source.bed_count,
--   source.created_at, source.updated_at, source.landing_timeStamp, source.insert_timeStamp,
--   source.insert_timeStame, source.updated_timeStamp
-- );

In [0]:
#to check the how many records have been written
record_count = bronze_df_2.count()
dbutils.notebook.exit(str(record_count))